In [11]:
import json
from copy import deepcopy
from tqdm import tqdm
import re
import os

In [12]:
# Load datasets accordingly to the subtask

# Select which file we should be working on
file_to_load = "../data/SOMD 2026/subtask 1/train_data.jsonl"
with open(file_to_load, 'r') as f:
    train_data = [json.loads(l) for l in list(f)]

with open("../data/SOMD 2026/subtask 1/train_labels.json", "r") as f:
    train_labels = json.load(f)

In [13]:
def inject_boundary_noise(train_data, noise_rate):
    """
    Add boundary errors with different error types.
    
    Error types:
    - Add word on right (40%)
    - Add word on left (30%)
    - Add words on both sides (15%)
    - Truncate right (10%)
    - Truncate left (5%)
    """
    new_train_data = deepcopy(train_data)
    
    import random
    n_to_modify = int(len(new_train_data) * noise_rate)
    indices_to_modify = set(random.sample(range(len(new_train_data)), n_to_modify))
    
    stats = {'add_right': 0, 
             'add_left': 0, 
             'add_both': 0, 
             'truncate_right': 0, 
             'truncate_left': 0, 
             'skipped': 0}
    
    for idx, mention in enumerate(tqdm(new_train_data, desc="Injecting boundary noise")):
        if idx not in indices_to_modify:
            continue
        
        sentence = mention["sentence"]
        start = mention["start"]
        end = mention["end"]
        current_mention = mention["mention"]
        
        # Randomly choose error type
        error_type = random.choices(
            ['add_right', 'add_left', 'add_both', 'truncate_right', 'truncate_left'],
            weights=[40, 30, 15, 10, 5]
        )[0]
        
        try:
            if error_type == 'add_right':
                # Add next word
                after_text = sentence[end:].strip()
                if after_text:
                    match = re.match(r"^(\s*)(\S+)", sentence[end:])
                    if match:
                        new_end = end + len(match.group(0))
                        mention["end"] = new_end
                        mention["mention"] = sentence[start:new_end]
                        stats['add_right'] += 1
                    else:
                        stats['skipped'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'add_left':
                # Add previous word
                before_text = sentence[:start].strip()
                if before_text:
                    match = re.search(r"(\S+)(\s*)$", sentence[:start])
                    if match:
                        prev_word_start = match.start()
                        mention["start"] = prev_word_start
                        mention["mention"] = sentence[prev_word_start:end]
                        stats['add_left'] += 1
                    else:
                        stats['skipped'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'add_both':
                # Add both previous and next word
                # (combine add_left and add_right logic)
                before = sentence[:start].strip()
                after = sentence[end:].strip()
                if before and after:
                    left_match = re.search(r"(\S+)(\s*)$", sentence[:start])
                    right_match = re.match(r"^(\s*)(\S+)", sentence[end:])
                    if left_match and right_match:
                        new_start = left_match.start()
                        new_end = end + len(right_match.group(0))
                        mention["start"] = new_start
                        mention["end"] = new_end
                        mention["mention"] = sentence[new_start:new_end]
                        stats['add_both'] += 1
                    else:
                        stats['skipped'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'truncate_right':
                # Remove last 1-2 characters
                if len(current_mention) > 3:
                    chars_to_remove = random.randint(1, min(2, len(current_mention) - 2))
                    mention["end"] = end - chars_to_remove
                    mention["mention"] = sentence[start:end - chars_to_remove]
                    stats['truncate_right'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'truncate_left':
                # Remove first 1-2 characters
                if len(current_mention) > 3:
                    chars_to_remove = random.randint(1, min(2, len(current_mention) - 2))
                    mention["start"] = start + chars_to_remove
                    mention["mention"] = sentence[start + chars_to_remove:end]
                    stats['truncate_left'] += 1
                else:
                    stats['skipped'] += 1
        
        except Exception as e:
            stats['skipped'] += 1
            continue
    
    print(f"\nBoundary noise injection stats:")
    for error_type, count in stats.items():
        print(f"  {error_type}: {count}")
    
    total_modified = sum(stats.values()) - stats['skipped']
    print(f"  Total modified: {total_modified}/{len(new_train_data)} ({total_modified/len(new_train_data):.1%})")
    
    return new_train_data


In [14]:
noise_rates = [0, 0.25, 0.50, 0.75, 1.0]

In [ ]:
for noise_rate in noise_rates:
    noisy_data = inject_boundary_noise(train_data, noise_rate = noise_rate)
    # Save dataset locally 
    filename = f"../../SOMD-2026/data/SOMD 2026/subtask 1/noisy_data/noisy_{noise_rate}_train_data_subtask_1.jsonl"
    with open(filename, 'w') as f:
        for item in noisy_data:
            json_record = json.dumps(item)
            f.write(json_record + '\n')

Injecting boundary noise: 100%|██████████| 2974/2974 [00:00<00:00, 7298923.40it/s]



Boundary noise injection stats:
  add_right: 0
  add_left: 0
  add_both: 0
  truncate_right: 0
  truncate_left: 0
  skipped: 0
  Total modified: 0/2974 (0.0%)


Injecting boundary noise: 100%|██████████| 2974/2974 [00:00<00:00, 701724.80it/s]



Boundary noise injection stats:
  add_right: 288
  add_left: 206
  add_both: 110
  truncate_right: 66
  truncate_left: 39
  skipped: 34
  Total modified: 709/2974 (23.8%)


Injecting boundary noise: 100%|██████████| 2974/2974 [00:00<00:00, 378684.28it/s]



Boundary noise injection stats:
  add_right: 588
  add_left: 433
  add_both: 203
  truncate_right: 145
  truncate_left: 57
  skipped: 61
  Total modified: 1426/2974 (47.9%)


Injecting boundary noise: 100%|██████████| 2974/2974 [00:00<00:00, 268422.46it/s]



Boundary noise injection stats:
  add_right: 920
  add_left: 613
  add_both: 318
  truncate_right: 188
  truncate_left: 97
  skipped: 94
  Total modified: 2136/2974 (71.8%)


Injecting boundary noise: 100%|██████████| 2974/2974 [00:00<00:00, 205893.64it/s]


Boundary noise injection stats:
  add_right: 1216
  add_left: 805
  add_both: 421
  truncate_right: 278
  truncate_left: 114
  skipped: 140
  Total modified: 2834/2974 (95.3%)
